In [ ]:
combined_embedding_model = ''
separate_embedding_model = ''


# run through validation set, get embeddings for both models

# get pairwise cosine similarities and plot (node level from proof traces)

# run with only one beam, get reconstruction, save to file with combined/separate prediction for a given sample
# (automate this and include for validation training??)


# You are an expert in Lean 3 theorem proving. You are tasked to evaluate the performance of two models which predict 
# the result from applying a tactic to a given goal. 
# Your evaluation criteria should consider the following:
# 1.        



In [ ]:
from models.end_to_end.tactic_models.tac_embed_separate.model import TransitionModel as SeparateModel
from models.end_to_end.tactic_models.tac_embed_large.model import TransitionModelLarge as CombinedModel



In [ ]:
# load from data module
def collate_fn(examples):
    goal = [ex["tactic"] + ex["theorem"] + '\n\n' + ex["goal"] for ex in examples]

    tokenized_goal = tokenizer(
        goal,
        padding="longest",
        max_length=int(3000),
        truncation=True,
        return_tensors="pt",
    )

    result = [ex["result"] for ex in examples]

    tokenized_result = tokenizer(
        result,
        padding="longest",
        max_length=2000,
        truncation=True,
        return_tensors="pt",
    )

    tactic = [ex["tactic"] for ex in examples]

    tokenized_tactic = tokenizer(
        tactic,
        padding="longest",
        max_length=2000,
        truncation=True,
        return_tensors="pt",
    )

    lens = tokenized_tactic.attention_mask.sum(dim=1)

    result_ids = tokenized_result.input_ids
    result_ids[result_ids == tokenizer.pad_token_id] = -100

    tactic_ids = tokenized_tactic.input_ids

    batch = {}
    batch["goal"] = goal
    batch["goal_ids"] = tokenized_goal.input_ids
    batch["goal_mask"] = tokenized_goal.attention_mask
    batch["result"] = result
    batch["result_ids"] = tokenized_result.input_ids
    batch["result_mask"] = tokenized_goal.attention_mask
    batch["tactic"] = tactic
    batch["tactic_lens"] = lens
    batch["tactic_ids"] = tactic_ids
    batch["tactic_mask"] = tokenized_tactic.attention_mask

    return batch


In [ ]:
from models.end_to_end.tactic_models.tac_embed_large.datamodule import TransitionDataModule


module = TransitionDataModule(model_name='sean-lamont/leandojo-lean3-reprover-novel-premises', batch_size=1,
                              eval_batch_size=1, max_seq_len=2000, num_workers=0, trace_files='runs/bestfs-novel-2',
                              replace='keep')

In [ ]:
module.trainer.global_rank

In [ ]:
module.setup()